In [ ]:
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from  tensorflow.python import keras
from tensorflow.python.keras import Sequential
import matplotlib.pyplot as plt
from tensorflow.python.keras.layers import Dense, Activation, Dropout, Flatten, Conv2D, MaxPooling2D,AveragePooling2D
from tensorflow.keras.layers import BatchNormalization
import numpy as np


with open('/content/gdrive/MyDrive/Colab Notebooks/OneShot/trainingwithoutchange/400/output.txt', 'w') as f:
  f.write("400 Resim")
  f.write("\n")

shape = (224,224,3)

#train
flat = np.load('/content/gdrive/MyDrive/Colab Notebooks/OneShot/trainingwithoutchange/400/flat400.npy')
gable = np.load('/content/gdrive/MyDrive/Colab Notebooks/OneShot/trainingwithoutchange/400/gable400.npy')
hip = np.load('/content/gdrive/MyDrive/Colab Notebooks/OneShot/trainingwithoutchange/400/hip400.npy')

flat_Y = np.zeros(400,dtype=int)
gable_Y = np.ones(400,dtype=int)
hip_Y = np.zeros(400,dtype=int)
hip_Y[:]=2

X_all = np.vstack((flat[1:],gable[1:],hip[1:]))
X_all = np.expand_dims(X_all, axis=4)

Y_all = np.hstack((flat_Y,gable_Y,hip_Y))

Y_temp = Y_all
res = np.zeros((Y_all.size,3))
res[np.arange(Y_all.size), Y_all] = 1
Y_all=res

X_all=np.squeeze(X_all)

#test
flat_test = np.load('/content/gdrive/MyDrive/Colab Notebooks/OneShot/Evoluation/flatevo.npy')
gable_test = np.load('/content/gdrive/MyDrive/Colab Notebooks/OneShot/Evoluation/gableevo.npy')
hip_test = np.load('/content/gdrive/MyDrive/Colab Notebooks/OneShot/Evoluation/hipevo.npy')

flat_Y_test = np.zeros(2400,dtype=int)
gable_Y_test = np.ones(2400,dtype=int)
hip_Y_test = np.zeros(2400,dtype=int)
hip_Y_test[:]=2

X_all_test = np.vstack((flat_test[1:],gable_test[1:],hip_test[1:]))
X_all_test = np.expand_dims(X_all_test, axis=4)

Y_all_test = np.hstack((flat_Y_test,gable_Y_test,hip_Y_test))

Y_temp_test = Y_all_test
res = np.zeros((Y_all_test.size,3))
res[np.arange(Y_all_test.size), Y_all_test] = 1
Y_all_test=res

X_all_test=np.squeeze(X_all_test)
tx,ty = Y_all.shape

ti = X_all.shape
with open('/content/gdrive/MyDrive/Colab Notebooks/OneShot/trainingwithoutchange/400/output.txt', 'a') as f:
  f.write(str(ti))
  f.write("\n")
print(ti)
tx,ty = Y_all.shape
print(tx)
with open('/content/gdrive/MyDrive/Colab Notebooks/OneShot/trainingwithoutchange/400/output.txt', 'a') as f:
  f.write(str(tx))
  f.write("\n")



(1200, 224, 224, 3)
1200


In [ ]:
model = Sequential()

# C1 Convolutional Layer
model.add(Conv2D(6, kernel_size=(5, 5), strides=(1, 1), activation='tanh', input_shape=(28,28,1), padding="same"))

# S2 Pooling Layer
model.add(AveragePooling2D(pool_size=(2, 2), strides=(1, 1), padding='valid'))

# C3 Convolutional Layer
model.add(Conv2D(16, kernel_size=(5, 5), strides=(1, 1), activation='tanh', padding='valid'))

# S4 Pooling Layer
model.add(AveragePooling2D(pool_size=(2, 2), strides=(2, 2), padding='valid'))

# C5 Fully Connected Convolutional Layer
model.add(Conv2D(120, kernel_size=(5, 5), strides=(1, 1), activation='tanh', padding='valid'))
#Flatten the CNN output so that we can connect it with fully connected layers
model.add(Flatten())

# FC6 Fully Connected Layer
model.add(Dense(84, activation='tanh'))

#Output Layer with softmax activation
model.add(Dense(3, activation='softmax'))

# Compile the model
model.compile(loss=keras.losses.categorical_crossentropy, optimizer='SGD', metrics=["accuracy"])

In [ ]:

# Model configuration
batch_size = 32
no_epochs = 100
no_classes = 3    
verbosity = 1

learning_rate = 0.0001

from sklearn import metrics
from sklearn.model_selection import KFold

Acc_all = []
Loss_all = []

Y_Pred_all = np.zeros(tx)
from numpy import argmax
 
# Fit data to model
history = model.fit(X_all, Y_all,
          batch_size=batch_size,
          epochs=no_epochs,
          verbose=verbosity)

score = model.evaluate(X_all_test, Y_all_test,batch_size=1, verbose=1)
print(f'Test loss: {score[0]} / Test accuracy: {score[1]}')

with open('/content/gdrive/MyDrive/Colab Notebooks/OneShot/trainingwithoutchange/2400/output.txt', 'a') as f:
  f.write(f'Test loss: ')
  f.write(str(score[0]))
  f.write("\n")
  f.write('Test accuracy: ')
  f.write(str(score[1]))
  f.write("\n")
  
Acc_all.append(score[1])
Loss_all.append(score[0])

Y_Pred = model.predict(X_all_test,batch_size=1,verbose=1)
Y_Pred = argmax(Y_Pred, axis=1)

print(np.average(Acc_all))
print(np.average(Loss_all))

with open('/content/gdrive/MyDrive/Colab Notebooks/OneShot/trainingwithoutchange/2400/output.txt', 'a') as f:
  f.write(str(np.average(Acc_all)))
  f.write("\n")
  f.write(str(np.average(Loss_all)))
  f.write("\n")
  
Y_True = Y_temp

import sklearn.metrics as metrics
acc = metrics.accuracy_score(Y_True, Y_Pred_all)
f1 = metrics.f1_score(Y_True,Y_Pred_all, average=None)
precision = metrics.precision_score(Y_True,Y_Pred_all, average=None)
recall = metrics.recall_score(Y_True,Y_Pred_all, average=None)
confusion = metrics.confusion_matrix(Y_True,Y_Pred_all)



print("TRAINING DATA RESULTS")
print("Accuracy:   "+str(acc))
print("F1:         "+str(f1))
print("Precision:  "+str(precision))
print("Recall:     "+str(recall))
print("Confusion:  ")
print(str(confusion))

with open('/content/gdrive/MyDrive/Colab Notebooks/OneShot/trainingwithoutchange/2400/output.txt', 'a') as f:
  f.write("\n")
  f.write("TRAINING DATA RESULTS")
  f.write("\n")
  f.write("Accuracy:   "+str(acc))
  f.write("\n")
  f.write("F1:         "+str(f1))
  f.write("\n")
  f.write("Precision:  "+str(precision))
  f.write("\n")
  f.write("Recall:     "+str(recall))
  f.write("\n")
  f.write("Confusion:  ")
  f.write(str(confusion))
  f.write("\n")

Y_True=Y_temp_test
acc = metrics.accuracy_score(Y_True, Y_Pred)
f1 = metrics.f1_score(Y_True,Y_Pred, average=None)
precision = metrics.precision_score(Y_True,Y_Pred, average=None)
recall = metrics.recall_score(Y_True,Y_Pred, average=None)
confusion = metrics.confusion_matrix(Y_True,Y_Pred)

print("TEST DATA RESULTS")
print("Accuracy:   "+str(acc))
print("F1:         "+str(f1))
print("Precision:  "+str(precision))
print("Recall:     "+str(recall))
print("Confusion:  ")
print(str(confusion))

with open('/content/gdrive/MyDrive/Colab Notebooks/OneShot/trainingwithoutchange/2400/output.txt', 'a') as f:
  f.write("\n")
  f.write("TEST DATA RESULTS")
  f.write("\n")
  f.write("Accuracy:   "+str(acc))
  f.write("\n")
  f.write("F1:         "+str(f1))
  f.write("\n")
  f.write("Precision:  "+str(precision))
  f.write("\n")
  f.write("Recall:     "+str(recall))
  f.write("\n")
  f.write("Confusion:  ")
  f.write(str(confusion))

Epoch 1/100


ValueError: ignored